# 03 — Dataset di Coppie per la Rete Siamese e Precomputazione Cache FCGR

## Scopo del Notebook
1. Costruire i pool bilanciati di **Validation** e **Test** per il task binario (`0` = Stabile/Healthy vs `1` = Hotspot/Disease-associated).
2. Precomputare su disco il file memory-mapped `fcgr_k6.npy` (`float32`, $64 \times 64$) per tutte le sequenze di Train, Validation e Test (con salvataggio di checkpoint ogni `5.000` sequenze e misurazione del tempo totale di generazione).
3. Confrontare il throughput (`pair/s` e `ms/batch`) tra calcolo FCGR *on-demand* e lettura da `memmap`.
4. Ispezionare visivamente le coppie `(x1, x2)` positive e negative con `plot_siamese_pair_feedback`.

In [ ]:
import json
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from numpy.lib.format import open_memmap

from pathlib import Path
import sys

# Rilevamento automatico Google Colab vs Ambiente Locale
COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / 'eccdna_disease_detection_metadata.tsv'
FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('META_PATH    :', META_PATH, '-> exists:', META_PATH.exists())
print('FASTA_PATH   :', FASTA_PATH, '-> exists:', FASTA_PATH.exists())

from src.fcgr import (
    load_fasta_sequences,
    compute_fcgr,
    normalize_fcgr,
    plot_siamese_pair_feedback,
)
from src.dataset import SiamesePairDataset, SiamesePairDatasetMemmap

## 1. Configurazione e Costruzione dei Pool Binari (Train / Val / Test)
Selezioniamo dal manifest binario il training set e campioniamo in modo bilanciato (`MAX_EVAL_SAMPLES_PER_CLASS = 2.500` per classe) i pool di Validation (`5.000` sequenze) e Test (`5.000` sequenze).

In [ ]:
RANDOM_STATE = 42
DEFAULT_K = 6  # FCGR 64x64
BATCH_SIZE = 64
POSITIVE_PAIR_PROBABILITY = 0.50
TRAIN_PAIRS_PER_EPOCH = 50_000
VAL_PAIRS = 10_000
TEST_PAIRS = 10_000
MAX_EVAL_SAMPLES_PER_CLASS = 2_500  # 2500 x 2 = 5000 seq per Val e 5000 seq per Test

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

metadata = pd.read_csv(MANIFEST_PATH, sep='\t', low_memory=False)
metadata['id'] = metadata['id'].astype(str)
metadata['class_id'] = metadata['class_id'].astype(int)

train_metadata = metadata[metadata['split_cluster'] == 'train'].copy().reset_index(drop=True)

def sample_balanced_eval_pool(df_split, max_per_class, seed):
    parts = []
    for cid in sorted(df_split['class_id'].unique()):
        sub = df_split[df_split['class_id'] == cid]
        parts.append(sub.sample(n=min(len(sub), max_per_class), replace=False, random_state=seed + int(cid)))
    return pd.concat(parts, ignore_index=True)

val_pair_pool = sample_balanced_eval_pool(metadata[metadata['split_cluster'] == 'val'], MAX_EVAL_SAMPLES_PER_CLASS, RANDOM_STATE + 100)
test_pair_pool = sample_balanced_eval_pool(metadata[metadata['split_cluster'] == 'test'], MAX_EVAL_SAMPLES_PER_CLASS, RANDOM_STATE + 200)

VAL_POOL_PATH = PROCESSED_DIR / 'siamese_val_pair_pool.tsv'
TEST_POOL_PATH = PROCESSED_DIR / 'siamese_test_pair_pool.tsv'
val_pair_pool.to_csv(VAL_POOL_PATH, sep='\t', index=False)
test_pair_pool.to_csv(TEST_POOL_PATH, sep='\t', index=False)

print(f'Train samples : {len(train_metadata):,}')
print(f'Val pool      : {len(val_pair_pool):,} ( salvato in {VAL_POOL_PATH.name} )')
print(f'Test pool     : {len(test_pair_pool):,} ( salvato in {TEST_POOL_PATH.name} )')
display(pd.DataFrame({
    'Train': train_metadata['class_id'].value_counts(),
    'Val Pool': val_pair_pool['class_id'].value_counts(),
    'Test Pool': test_pair_pool['class_id'].value_counts(),
}))

## 2. Caricamento Sequenze FASTA e Precomputazione Cache `fcgr_k6.npy`
Carichiamo dal file FASTA tutte le sequenze necessarie (`Train + Val Pool + Test Pool`) e precomputiamo su disco il tensore memory-mapped `fcgr_k6.npy`, registrando il tempo e lo spazio occupato dalla Baseline.

In [ ]:
required_fcgr_metadata = (
    pd.concat([train_metadata, val_pair_pool, test_pair_pool], ignore_index=True)
    .drop_duplicates(subset='id')
    .sort_values('id')
    .reset_index(drop=True)
)
required_fcgr_metadata.insert(0, 'fcgr_row', np.arange(len(required_fcgr_metadata), dtype=np.int64))

FCGR_CACHE_DIR = PROCESSED_DIR / 'fcgr_cache'
FCGR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

FCGR_MEMMAP_PATH = FCGR_CACHE_DIR / f'fcgr_k{DEFAULT_K}.npy'
FCGR_INDEX_PATH = FCGR_CACHE_DIR / f'fcgr_k{DEFAULT_K}_index.tsv'
FCGR_META_PATH = FCGR_CACHE_DIR / f'fcgr_k{DEFAULT_K}_metadata.json'
FCGR_TEMP_PATH = FCGR_CACHE_DIR / f'fcgr_k{DEFAULT_K}_tmp.npy'
FCGR_PROGRESS_PATH = FCGR_CACHE_DIR / f'fcgr_k{DEFAULT_K}_progress.json'

N_FCGR_SAMPLES = len(required_fcgr_metadata)
FCGR_SIDE = 2 ** DEFAULT_K
expected_mb = (N_FCGR_SAMPLES * FCGR_SIDE * FCGR_SIDE * 4) / (1024 ** 2)

print(f'Sequenze uniche totali da codificare : {N_FCGR_SAMPLES:,}')
print(f'Risoluzione FCGR (k={DEFAULT_K})             : {FCGR_SIDE}x{FCGR_SIDE}')
print(f'Dimensione file memmap (float32)     : {expected_mb:.2f} MB ({expected_mb/1024:.3f} GB)')

In [ ]:
CHECKPOINT_EVERY = 5_000
fcgr_ids = required_fcgr_metadata['id'].tolist()

if FCGR_MEMMAP_PATH.exists() and FCGR_INDEX_PATH.exists():
    existing_idx = pd.read_csv(FCGR_INDEX_PATH, sep='\t', dtype={'id': str})
    if set(existing_idx['id']) == set(fcgr_ids):
        print('Cache FCGR già presente e allineata con il manifest corrente. Salto la rigenerazione.')
         regenerate = False
    else:
        print('Il manifest è cambiato rispetto alla vecchia cache: rigenerazione in corso...')
        regenerate = True
else:
    regenerate = True

if regenerate:
    print('Caricamento sequenze dal file FASTA...')
    t_fasta = time.perf_counter()
    sequence_store = load_fasta_sequences(FASTA_PATH, fcgr_ids)
    print(f'Caricate {len(sequence_store):,} sequenze in {time.perf_counter() - t_fasta:.1f} s')
    assert len(sequence_store) == N_FCGR_SAMPLES, 'Alcuni ID non sono stati trovati nel FASTA!'

    fcgr_memmap_write = open_memmap(
        FCGR_TEMP_PATH,
        mode='w+',
        dtype=np.float32,
        shape=(N_FCGR_SAMPLES, FCGR_SIDE, FCGR_SIDE),
    )

    t_gen_start = time.perf_counter()
    for row_idx, sample_id in enumerate(fcgr_ids):
        seq = sequence_store[sample_id]
        counts, n_valid, _ = compute_fcgr(seq, DEFAULT_K, return_distinct=True)
        fcgr_memmap_write[row_idx] = normalize_fcgr(counts)

        if (row_idx + 1) % CHECKPOINT_EVERY == 0 or (row_idx + 1) == N_FCGR_SAMPLES:
            fcgr_memmap_write.flush()
            elapsed_s = time.perf_counter() - t_gen_start
            rate = (row_idx + 1) / max(elapsed_s, 1e-6)
            print(f'  [{row_idx + 1:6,d} / {N_FCGR_SAMPLES:,}] ({rate:.1f} seq/s, elapsed: {elapsed_s:.1f}s)')

    total_gen_seconds = time.perf_counter() - t_gen_start
    fcgr_memmap_write.flush()
    del fcgr_memmap_write
    FCGR_TEMP_PATH.replace(FCGR_MEMMAP_PATH)

    required_fcgr_metadata[['fcgr_row', 'id', 'class_id', 'disease_clean', 'split_cluster']].to_csv(
        FCGR_INDEX_PATH, sep='\t', index=False
    )
    with open(FCGR_META_PATH, 'w', encoding='utf-8') as f:
        json.dump({
            'k': DEFAULT_K,
            'n_samples': N_FCGR_SAMPLES,
            'shape': [N_FCGR_SAMPLES, FCGR_SIDE, FCGR_SIDE],
            'dtype': 'float32',
            'generation_seconds': total_gen_seconds,
            'ms_per_sequence': (total_gen_seconds * 1000.0) / N_FCGR_SAMPLES,
        }, f, indent=4)
    print(f'Precomputazione completata in {total_gen_seconds:.1f} s ({total_gen_seconds*1000/N_FCGR_SAMPLES:.2f} ms/seq).')

## 3. Verifica del DataLoader e Ispezione Visiva delle Coppie Siamesi (`plot_siamese_pair_feedback`)
Apriamo `fcgr_k6.npy` in modalità `mmap_mode='r'` e visualizziamo una coppia **Positiva** (stessa classe binaria) e una **Negativa** (Stabile vs Hotspot) pronte per la Rete Siamese.

In [ ]:
fcgr_memmap = np.load(FCGR_MEMMAP_PATH, mmap_mode='r')
fcgr_index = pd.read_csv(FCGR_INDEX_PATH, sep='\t', dtype={'id': str})
id_to_fcgr_row = dict(zip(fcgr_index['id'], fcgr_index['fcgr_row']))

val_pair_dataset_mm = SiamesePairDatasetMemmap(
    metadata=val_pair_pool,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    k=DEFAULT_K,
    pairs_per_epoch=VAL_PAIRS,
    positive_probability=POSITIVE_PAIR_PROBABILITY,
    seed=RANDOM_STATE + 10_000,
    deterministic=True,
)

ID_TO_CLASS = {0: 'Stable (0)', 1: 'Hotspot (1)'}

# Cerchiamo e plottiamo 1 coppia positiva e 1 coppia negativa
shown_targets = set()
for i in range(100):
    sample_pair = val_pair_dataset_mm[i]
    t_val = int(round(sample_pair['target'].item()))
    if t_val not in shown_targets:
        shown_targets.add(t_val)
        c1_name = ID_TO_CLASS[int(sample_pair['class1'])]
        c2_name = ID_TO_CLASS[int(sample_pair['class2'])]
        plot_siamese_pair_feedback(
            x1=sample_pair['x1'],
            x2=sample_pair['x2'],
            target=t_val,
            label1=f"{sample_pair['id1']} [{c1_name}]",
            label2=f"{sample_pair['id2']} [{c2_name}]",
        )
    if len(shown_targets) == 2:
        break